# 词袋模型：最朴素的文本表示
> 难度标签：高级 | 预计时长：10-20分钟 | 前置知识：无学习经验


> 所属模块：08_自然语言处理 | 源文件：词袋模型.py | 核心功能：CountVectorizer、词频统计、文本向量化

## 概述

词袋模型（Bag of Words）把文本表示为词频向量——忽略词序，只统计每个词出现的次数。简单但有效，是文本分类的经典基线。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import numpy as np
from sklearn.feature_extraction.text import CountVectorizer

## 数学原理

### 1. 词袋模型的数学表示

词袋模型（Bag of Words, BoW）将文本表示为**词频向量**，忽略词序和语法结构。

给定词汇表 $V = \{w_1, w_2, \ldots, w_{|V|}\}$，文档 $d$ 被映射为向量：

$$\mathbf{x}_d = (c(w_1, d), c(w_2, d), \ldots, c(w_{|V|}, d))$$

其中 $c(w_i, d)$ 是词 $w_i$ 在文档 $d$ 中的出现次数。

### 2. 文档-词频矩阵

语料库 $D = \{d_1, d_2, \ldots, d_n\}$ 的词袋表示为矩阵：

$$X \in \mathbb{R}^{n \times |V|}, \quad X_{ij} = c(w_j, d_i)$$

代码中 `CountVectorizer().fit_transform(corpus)` 输出的稀疏矩阵即为此矩阵。

### 3. 中文分词的必要性

中文没有天然的词边界，需要先分词：

$$\text{"机器学习是人工智能的基础"} \to \text{"机器 学习 是 人工智能 的 基础"}$$

代码中 `jieba.cut()` 执行中文分词，将连续字符序列切分为词序列。

### 4. 词袋模型的变体

**二值词袋**（Binary BoW）：只记录词是否出现

$$X_{ij} = \mathbb{I}[c(w_j, d_i) > 0]$$

**归一化词袋**：除以文档总词数，消除文档长度影响

$$\hat{X}_{ij} = \frac{c(w_j, d_i)}{\sum_{k} c(w_k, d_i)}$$

### 5. 参数控制

代码中 `CountVectorizer` 的关键参数：

| 参数 | 数学含义 | 作用 |
|------|----------|------|
| `min_df` | $c(w, D) < \text{min\_df}$ 的词被忽略 | 去除极低频词（噪声） |
| `max_df` | 文档频率 $> \text{max\_df}$ 的词被忽略 | 去除极高频词（停用词） |
| `ngram_range` | 包含 $n$-gram 的范围 | 捕捉短语，如 "机器_学习" |
| `max_features` | 保留 TF 最高的 $k$ 个词 | 控制维度 |

### 6. N-gram 扩展

将相邻 $n$ 个词组合作为特征：

$$\text{bigram}: \text{"机器 学习 是"} \to \{\text{机器学习}, \text{学习是}\}$$

词汇表大小从 $|V|$ 扩展为 $|V|^n$（理论上），实际通过 `max_features` 限制。

### 7. 词袋模型的局限

- **维度灾难**：$|V|$ 通常为数万到数十万
- **稀疏性**：每个文档只包含少量词，向量高度稀疏
- **无语义**：不考虑词序、同义词、多义词
- **无权重**：所有词同等对待，不区分重要性

TF-IDF 和 Word2Vec 分别从权重和语义两个角度改进。

### 8. 代码与数学的对应

| 代码 | 数学含义 |
|------|----------|
| `CountVectorizer()` | 构建词汇表 $V$，计算 $X_{ij} = c(w_j, d_i)$ |
| `X.shape = (n, |V|)` | 文档-词频矩阵维度 |
| `get_feature_names_out()` | 返回词汇表 $V = [w_1, \ldots, w_{|V|}]$ |
| `toarray()` | 稀疏向量 $\to$ 稠密向量 |
| `min_df=2` | 过滤 $c(w, D) < 2$ 的低频词 |
| `ngram_range=(1,2)` | 同时使用 unigram 和 bigram |

### 1. 示例文档

运行 1. 示例文档 的代码，观察算法在该环节的行为。

In [ ]:
corpus = [
    "机器学习是人工智能的基础",
    "深度学习是机器学习的子领域",
    "自然语言处理用到深度学习",
    "计算机视觉也依赖机器学习",
# --- 继续 ---
    "强化学习是另一个重要方向",
]
print("=== 语料库 ===")
for i, doc in enumerate(corpus):
    print(f"  文档{i+1}: {doc}")

### 2. 中文分词

运行 2. 中文分词 的代码，观察算法在该环节的行为。

In [ ]:
try:
    import jieba
    corpus_cut = [" ".join(jieba.cut(doc)) for doc in corpus]
except ImportError:
    # 简单按字符分词
    corpus_cut = [" ".join(doc) for doc in corpus]

print("\n=== 分词后 ===")
for i, doc in enumerate(corpus_cut):
    print(f"  文档{i+1}: {doc}")

### 3. CountVectorizer

运行 3. CountVectorizer 的代码，观察算法在该环节的行为。

In [ ]:
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(corpus_cut)

print(f"\n=== 词袋矩阵 ===")
print(f"矩阵形状: {X.shape} (文档数 × 词汇表大小)")
print(f"词汇表: {vectorizer.get_feature_names_out()}")

# 稀疏矩阵转稠密显示
print(f"\n词频矩阵:")
for i, doc in enumerate(corpus_cut):
    freq = X[i].toarray().flatten()
    non_zero = [(vectorizer.get_feature_names_out()[j], freq[j]) for j in range(len(freq)) if freq[j] > 0]
    print(f"  文档{i+1}: {non_zero}")

### 4. 参数设置

探索关键超参数对模型行为的影响。

In [ ]:
print("\n=== CountVectorizer 参数 ===")

# min_df: 忽略出现次数低于此阈值的词
vec_min_df = CountVectorizer(min_df=2)
X_min = vec_min_df.fit_transform(corpus_cut)
print(f"min_df=2: 词汇表={vec_min_df.get_feature_names_out()}")

# max_df: 忽略出现频率高于此比例的词
vec_max_df = CountVectorizer(max_df=0.8)
X_max = vec_max_df.fit_transform(corpus_cut)
print(f"max_df=0.8: 词汇表={vec_max_df.get_feature_names_out()}")

# max_features: 限制词汇表大小
vec_top = CountVectorizer(max_features=5)
X_top = vec_top.fit_transform(corpus_cut)
print(f"max_features=5: 词汇表={vec_top.get_feature_names_out()}")

### 5. N-gram

运行 5. N-gram 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== N-gram ===")
# unigram
vec_1 = CountVectorizer(ngram_range=(1, 1))
X_1 = vec_1.fit_transform(corpus_cut)
print(f"Unigram (1,1): {len(vec_1.vocabulary_)} 个特征")
print(f"  特征: {vec_1.get_feature_names_out()}")

# bigram
vec_2 = CountVectorizer(ngram_range=(1, 2))
X_2 = vec_2.fit_transform(corpus_cut)
print(f"Bigram (1,2): {len(vec_2.vocabulary_)} 个特征")

### 6. 词频 vs TF-IDF

运行 6. 词频 vs TF-IDF 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 词袋模型的局限 ===")
print("1. 高频词主导（如'的'、'是'），需要停用词过滤")
print("2. 不考虑词序（'学习机器'和'机器学习'相同）")
print("3. 所有词权重相同（'的'和'人工智能'同等重要）")
print("4. 改进: TF-IDF（考虑词的区分度）")

### 7. 实际应用示例

运行 7. 实际应用示例 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 词袋模型用于分类 ===")
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import cross_val_score

# 模拟文本分类
docs = [
    "机器学习 深度学习 神经网络 算法",
    "自然语言处理 文本 分词 语义",
    "图像识别 卷积神经网络 视觉 目标检测",
    "股票 基金 投资 市场 经济",
# --- 继续 ---
    "足球 比赛 冠军 联赛 球员",
] * 10  # 重复增加样本数
labels = [0, 0, 0, 1, 2] * 10

vec = CountVectorizer()
X = vec.fit_transform(docs)
nb = MultinomialNB()
scores = cross_val_score(nb, X, labels, cv=3, scoring="accuracy")
print(f"词袋 + 朴素贝叶斯 CV准确率: {scores.mean():.4f}")

print("\n=== 词袋模型要点 ===")
print("- 简单高效，是文本分类的常用 baseline")
print("- 适合：文本分类、信息检索、主题建模")
print("- 不适合：需要理解语义或词序的任务")

## 关键代码解释

```python
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer(max_features=5000)
X = vectorizer.fit_transform(texts)
```

## 注意事项

1. 丢失词序信息："狗咬人" 和 "人咬狗" 表示相同
2. 高维稀疏矩阵
3. max_features 控制词汇表大小

## 总结与延伸

以上代码展示了 **词袋模型** 的完整流程。

**解读要点**：
- 观察 **词汇表大小** 和 **向量维度** 是否合理
- 检查相似词查询结果是否符合语义直觉
- 关注分类任务中的 **TF-IDF 权重** 分布

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **N-gram**：保留局部词序
- **TF-IDF**：词袋的加权版本
- **Word2Vec/BERT**：稠密向量表示
